<a href="https://colab.research.google.com/github/AlfredaDhaifullah04/Filament_PWL/blob/master/PCVK_M5_W5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODUL 5 – Histogram
## Histogram, Histogram Equalization, dan Dithering

**Mata Kuliah:** Pengolahan Citra dan Visi Komputer
**Minggu:** 5   
**Nama:** [Alfreda Dhaifullah Mahezwara]  
**NIM:** [2441107020219]


In [ ]:
#===== SEL IDENTITAS JANGAN DIHAPUS, HARUS SEL PERTAMA =====
import sys, platform, hashlib, datetime, zoneinfo

NAMA = 'Alfreda Dhaifullah Mahezwara'
NIM = '244107020219'

N = int(NIM[-3:])
PARAM = {
    'bins': 2**((N % 4) + 3),
    'clip': round(1.0 + (N % 5) * 0.5, 1),
    'tile': (N % 4) + 2,
    'L': 2**((N % 3) + 1)
}

waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))
print('NAMA   :', NAMA)
print('NIM    :', NIM, 'N =', N)
for k, v in PARAM.items():
    print('%-6s:' % k, v)
print('WAKTU  :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))

kunci = NIM + ' ' + waktu.strftime('%Y-%m-%d')
print('TOKEN  :', hashlib.sha256(kunci.encode()).hexdigest()[:16])
print('SISTEM :', sys.version.split()[0], platform.platform())


In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import glob

from google.colab import drive
drive.mount('/content/drive')

# Ubah path ini menyesuaikan dengan lokasi dataset citra Anda
CONTOH = '/content/drive/MyDrive/PCVK/Images'
BASE = '/content/drive/MyDrive/PCVK/Images/' + NIM


## E-1 PERCOBAAN HISTOGRAM


In [ ]:
# Fungsi Histogram Manual
def histogram_manual(img, L=256):
    h = np.zeros(L, dtype=np.int64)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            h[img[y, x]] += 1
    return h


In [ ]:
# Pertanyaan Praktikum E1 - 1: Bandingkan Numpy, cv2, dan Manual
img_lena = cv.imread(CONTOH + '/lena.jpg', cv.IMREAD_GRAYSCALE)
if img_lena is not None:
    # Manual
    hist_man = histogram_manual(img_lena)

    # Numpy
    hist_np, _ = np.histogram(img_lena.flatten(), bins=256, range=[0, 256])

    # OpenCV
    hist_cv = cv.calcHist([img_lena], [0], None, [256], [0, 256]).flatten()

    selisih_man_np = np.max(np.abs(hist_man - hist_np))
    selisih_man_cv = np.max(np.abs(hist_man - hist_cv))

    print("Selisih Manual vs Numpy (Lena):", selisih_man_np)
    print("Selisih Manual vs OpenCV (Lena):", selisih_man_cv)
else:
    print("Gambar lena.jpg tidak ditemukan. Pastikan path CONTOH sudah benar.")

# Silakan ubah kode di atas untuk citra KTM1b.jpg milik Anda!


In [ ]:
# Pertanyaan Praktikum E1 - 2 & 3
# Plot Histogram Ternormalisasi p(j) dan Kurva CDF untuk KTM
# (Tuliskan implementasinya dengan memanggil cv.calcHist dan np.cumsum di sini)
# Contoh pemanggilan untuk 1 gambar (silakan dikembangkan untuk KTM1a - KTM1d):

if img_lena is not None:
    h = hist_man / img_lena.size  # Normalisasi
    cdf = np.cumsum(h)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(h, color='black')
    plt.title('Normalized Histogram')

    plt.subplot(1, 2, 2)
    plt.plot(cdf, color='red')
    plt.title('CDF')
    plt.show()


## E-2 PERCOBAAN HISTOGRAM EQUALIZATION


In [ ]:
# Pertanyaan Praktikum E2 - 2: EqualizeHist vs Manual
img_lena_lc = cv.imread(CONTOH + '/lena_lc.jpg', cv.IMREAD_GRAYSCALE)
if img_lena_lc is not None:
    # OpenCV Equalization
    eq_cv = cv.equalizeHist(img_lena_lc)

    # Manual Equalization
    h = histogram_manual(img_lena_lc)
    cdf = np.cumsum(h / img_lena_lc.size)
    sk = np.round(cdf * 255).astype(np.uint8)
    eq_man = sk[img_lena_lc]

    selisih_eq = np.max(np.abs(eq_cv.astype(int) - eq_man.astype(int)))
    print("Selisih Maksimum Manual HE vs OpenCV HE:", selisih_eq)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1), plt.imshow(eq_cv, cmap='gray'), plt.title('OpenCV HE')
    plt.subplot(1, 2, 2), plt.imshow(eq_man, cmap='gray'), plt.title('Manual HE')
    plt.show()


In [ ]:
# Ekualisasi Citra Berwarna (Kanal BGR vs YCrCb, HSV, LAB)
bgr = cv.imread(CONTOH + '/lena.jpg')
if bgr is not None:
    # 1. Ekualisasi per kanal (B, G, R)
    kanal = list(cv.split(bgr))
    for i in range(3):
        kanal[i] = cv.equalizeHist(kanal[i])
    he_rgb = cv.merge(kanal)

    # 2. Kanal Y (YCrCb)
    ycrcb = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
    y, cr, cb = cv.split(ycrcb)
    he_y = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)

    # 3. Kanal V (HSV)
    hsv = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)
    h_c, sat, v = cv.split(hsv)
    he_v = cv.cvtColor(cv.merge([h_c, sat, cv.equalizeHist(v)]), cv.COLOR_HSV2BGR)

    # 4. Kanal L (LAB)
    lab = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)
    l_c, a_c, b_c = cv.split(lab)
    he_l = cv.cvtColor(cv.merge([cv.equalizeHist(l_c), a_c, b_c]), cv.COLOR_LAB2BGR)

    judul = ['Asli', 'Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L']
    citra = [bgr, he_rgb, he_y, he_v, he_l]

    plt.figure(figsize=(18, 4))
    for i, (c, t) in enumerate(zip(citra, judul), 1):
        plt.subplot(1, 5, i)
        plt.imshow(cv.cvtColor(c, cv.COLOR_BGR2RGB))
        plt.title(f"{NIM} - {t}", fontsize=9)
        plt.axis('off')
    plt.show()


In [ ]:
# Fungsi Menghitung Pergeseran Hue
def geser_hue(asli, hasil):
    h1 = cv.cvtColor(asli, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    h2 = cv.cvtColor(hasil, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    d = np.abs(h1 - h2)
    d = np.minimum(d, 180 - d) # Hue melingkar 0-179
    return d.mean()

if bgr is not None:
    print('%-22s %16s %16s' % ('Cara', 'geser Hue (der)', 'rerata terang'))
    for t, c in zip(judul, citra):
        print('%-22s %16.2f %16.1f' % (t, geser_hue(bgr, c) * 2, cv.cvtColor(c, cv.COLOR_BGR2GRAY).mean()))


In [ ]:
# CLAHE (Contrast Limited Adaptive Histogram Equalization)
if img_lena_lc is not None:
    clahe = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
    res_clahe = clahe.apply(img_lena_lc)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1), plt.imshow(img_lena_lc, cmap='gray'), plt.title('Asli Lena LC')
    plt.subplot(1, 2, 2), plt.imshow(res_clahe, cmap='gray'), plt.title(f"{NIM} - CLAHE clip {PARAM['clip']}")
    plt.show()


## E-3 TUGAS PRAKTIKUM DITHERING


In [ ]:
# Fungsi Dithering Floyd-Steinberg
def floyd_steinberg(gray, L=2):
    img = gray.astype(np.float32).copy()
    step = 255.0 / (L - 1)
    H, W = img.shape
    for y in range(H):
        for x in range(W):
            lama = img[y, x]
            baru = np.round(lama / step) * step
            img[y, x] = baru
            error = lama - baru
            if x + 1 < W:
                img[y, x + 1] += error * 7/16
            if x > 0 and y + 1 < H:
                img[y + 1, x - 1] += error * 3/16
            if y + 1 < H:
                img[y + 1, x] += error * 5/16
            if x + 1 < W and y + 1 < H:
                img[y + 1, x + 1] += error * 1/16
    return np.clip(img, 0, 255).astype(np.uint8)


In [ ]:
if img_lena is not None:
    # Dithering Langsung
    dither_langsung = floyd_steinberg(img_lena, L=PARAM['L'])

    # Ekualisasi dulu lalu dithering
    eq_lena = cv.equalizeHist(img_lena)
    dither_eq = floyd_steinberg(eq_lena, L=PARAM['L'])

    plt.figure(figsize=(20, 10))
    plt.subplot(1, 4, 1), plt.imshow(img_lena, cmap='gray'), plt.title(f'{NIM} - lena.jpg asli')
    plt.subplot(1, 4, 2), plt.imshow(dither_langsung, cmap='gray'), plt.title(f"{NIM} - dithering langsung, L={PARAM['L']}")
    plt.subplot(1, 4, 3), plt.imshow(eq_lena, cmap='gray'), plt.title(f'{NIM} - hasil ekualisasi')
    plt.subplot(1, 4, 4), plt.imshow(dither_eq, cmap='gray'), plt.title(f'{NIM} - ekualisasi lalu dithering')
    plt.show()
